In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
# совместимый версии
!pip install torchtext=='0.18.0' torch=='2.3.0' torchdata=='0.9.0' portalocker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 786.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 106.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 795.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [3]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torchtext import datasets
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchtext.data import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

In [4]:
train_data = datasets.IMDB(split='train')
test_data = datasets.IMDB(split='test')

In [5]:
train = DataLoader(train_data, batch_size=32, shuffle=True)
test = DataLoader(test_data, batch_size=32)

In [6]:
tokenizer = get_tokenizer("basic_english")

In [7]:
# проверяем какой столбец первым приходит(класс или данные)
for i in list(train_data)[:5]: # берём только 5 первых
    print(i)

(1, "This film is based on the novel by John Fante. Could someone please tell me why? I see absolutely no reason why this fine book should be adapted in this way. If you want to make a romantic melodramatic Hollywood production with Colin Farell and Selma Hayek, then how could you possibly make a connection to Ask The Dust (the novel)? -And if you wanted to make this story into a film, then why would you want to make it into a romantic melodramatic Hollywood production with Colin Farell and Selma Hayek? I don't get it.<br /><br />The adaptation of the story is poorly made, and if you have read the book and liked it, I'm almost sure you won't like what Towne did with it. <br /><br />In the beginning of the film you'll maybe find the casting odd, the acting bad and the cinematography just a bit overdone. But you hope for the best. I really hoped a lot during this film. I actually wanted it to be good. But it only gets worse, and it is as simple as that: Whether you read Fantes novel or n

In [8]:
def text_to_token(data):
    for label, text in data:
        yield tokenizer(text)

In [9]:
vocab = build_vocab_from_iterator(text_to_token(train_data), specials=["<unk>"])

In [10]:
vocab.set_default_index(vocab["<unk>"])

In [11]:
def change_text(x):
    return [vocab[i] for i in tokenizer(x)]

change_label = lambda label: 1 if label == 2 else 0

In [12]:
def collate_batch(batch):
    labels, texts = [], []
    for label, text in batch:
        labels.append(change_label(label))
        texts.append(torch.tensor(change_text(text), dtype=torch.int64))
    labels = torch.tensor(labels, dtype=torch.int64)
    texts = nn.utils.rnn.pad_sequence(texts, batch_first=True)
    return texts, labels

In [13]:
train_dataloader = DataLoader(list(train_data), batch_size=32, shuffle=True, collate_fn=collate_batch)
test_dataloader = DataLoader(list(test_data), batch_size=32, collate_fn=collate_batch)

In [14]:
class SentimentModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=64, output_dim=2, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.embedding(x)
        out, (h_n, c_n) = self.lstm(x)
        x = self.dropout(h_n[-1]) # берем скрытое состояние h_n
        return self.fc(x)

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [16]:
model_Sentiment = SentimentModel(len(vocab)).to(device)

In [17]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_Sentiment.parameters(), lr=0.001)

[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.3.0
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [19]:
for epoch in range(30):
    model_Sentiment.train()
    total_loss = 0
    for texts, labels in train_dataloader:
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        pred = model_Sentiment(texts)
        loss = loss_fn(pred, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Эпоха: {epoch + 1} - Потери: {round(total_loss, 2)}')

Эпоха: 1 - Потери: 534.63
Эпоха: 2 - Потери: 533.24
Эпоха: 3 - Потери: 531.98
Эпоха: 4 - Потери: 532.08
Эпоха: 5 - Потери: 506.4
Эпоха: 6 - Потери: 413.98
Эпоха: 7 - Потери: 353.51
Эпоха: 8 - Потери: 325.02
Эпоха: 9 - Потери: 272.7
Эпоха: 10 - Потери: 215.57
Эпоха: 11 - Потери: 184.67
Эпоха: 12 - Потери: 150.02
Эпоха: 13 - Потери: 147.02
Эпоха: 14 - Потери: 119.95
Эпоха: 15 - Потери: 88.31
Эпоха: 16 - Потери: 76.32
Эпоха: 17 - Потери: 71.75
Эпоха: 18 - Потери: 66.43
Эпоха: 19 - Потери: 61.74
Эпоха: 20 - Потери: 44.15
Эпоха: 21 - Потери: 47.89
Эпоха: 22 - Потери: 39.88
Эпоха: 23 - Потери: 41.26
Эпоха: 24 - Потери: 26.14
Эпоха: 25 - Потери: 20.4
Эпоха: 26 - Потери: 24.73
Эпоха: 27 - Потери: 17.69
Эпоха: 28 - Потери: 12.06
Эпоха: 29 - Потери: 11.23
Эпоха: 30 - Потери: 18.5


In [20]:
model_Sentiment.eval()
correct, total = 0, 0

with torch.no_grad():
    for x_batch, y_batch in test_dataloader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        y_pred = model_Sentiment(x_batch)
        pred = torch.argmax(y_pred, dim=1)

        correct += (pred == y_batch).sum().item()
        total += y_batch.size(0)

accuracy = correct * 100 / total
print(f'Точность предположения модели: {accuracy:.2f}%')

Точность предположения модели: 83.54%


In [22]:
def evaluate_accuracy(dataloader):
    model_Sentiment.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x_batch, y_batch in dataloader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            y_pred = model_Sentiment(x_batch)
            correct += (y_pred.argmax(1) == y_batch).sum().item()
            total += y_batch.size(0)
    return correct / total * 100

print(f'train score: {evaluate_accuracy(train_dataloader):.2f}%')
print(f'test score: {evaluate_accuracy(test_dataloader):.2f}%')

train score: 99.70%
test score: 83.54%


In [24]:
from google.colab import files

torch.save(vocab, 'vocab_Sentiment_IMDB_DatasetOf_50K_MovieReviews.pth')
torch.save(model_Sentiment.state_dict(), 'model_Sentiment_IMDB_DatasetOf_50K_MovieReviews.pth')

files.download('vocab_Sentiment_IMDB_DatasetOf_50K_MovieReviews.pth')
files.download('model_Sentiment_IMDB_DatasetOf_50K_MovieReviews.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>